# Task 3.1: Exploratory Data Analysis (EDA) and Preprocessing Pipeline
## AuraCart Retail Analytics - Production-Grade E-Commerce Analytics

This notebook encapsulates all statistical visualizations, feature engineering logic, correlation analyses, and the final serialization of the Scikit-learn preprocessing pipeline objects as required by the Project Specification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# Set visual style
sns.set_theme(style="whitegrid")

## 1. Data Characterization and Loading
We utilize the 'E-commerce Customer Order Behavior Dataset' from Hugging Face hub.

In [ ]:
print("Loading dataset...")
dataset = load_dataset("millat/e-commerce-orders")
df = pd.DataFrame(dataset['train'])
df.head()

## 2. Exploratory Data Analysis (EDA)
### 2.1 Analysis of Continuous Variables
We examine the distribution of 'price' and 'quantity' to detect skewness and outliers.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], kde=True, color='blue')
plt.title('Distribution of Transaction Price')
plt.show()

### 2.2 Feature Correlation Analysis
We generate a correlation matrix to examine relationships between numerical features.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

### 2.3 Categorical Variable Analysis
We visualize the distribution of delivery status and customer segments to identify class imbalances.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.countplot(x='delivery_status', data=df, ax=ax[0])
ax[0].set_title('Delivery Status Distribution')
sns.countplot(x='customer_segment', data=df, ax=ax[1])
ax[1].set_title('Customer Segment Distribution')
plt.show()

## 3. Preprocessing Pipeline
We build a reproducible Scikit-learn Pipeline for categorical encoding and feature scaling.

In [ ]:
# Drop non-informative IDs
X = df.drop(['order_id', 'customer_id', 'product_id', 'price'], axis=1)

numeric_features = ['quantity']
categorical_features = ['category', 'payment_method', 'device_type', 'channel']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

full_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
print("Preprocessing pipeline successfully built.")

# Save the preprocessing artifact
joblib.dump(full_pipeline, '../artifacts/preprocessor.joblib')
print("Artifact saved to artifacts/preprocessor.joblib")